##### Copyright 2026 White Circle.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Fine-tune Gemma 4 with Halo and SGLang

Cookbook author: White Circle ([hello@whitecircle.com](mailto:hello@whitecircle.com)).

Halo extends Hugging Face training with distributed execution. It keeps native model code and checkpoint formats, so saved models load with standard `from_pretrained`.

When a model does not scale well with stock TRL, Halo adds expert, context, tensor, and expert-tensor parallelism without a Megatron-LM port. It runs on one GPU or across multi-node EFA and InfiniBand clusters.

Halo also includes a BF16 optimizer, fused kernels, and methods for alignment and reinforcement learning. A new model family usually needs a small wrapper around its MoE or attention block.

This guide runs ten SFT steps and ten environmental GRPO steps on Gemma 4 26B-A4B. Halo trains the model, while SGLang generates the GRPO rollouts. The guide uses public White Circle container images and does not build an image locally.

## Hardware and runtime

Run this notebook on a Linux host with Docker and the NVIDIA Container Toolkit. A standard hosted Colab runtime does not provide this GPU topology. You can connect Colab to a custom runtime.

The setup cell selects the training image from the GPU compute capability. It uses the following recommended layouts:

| GPU type | SFT layout | GRPO layout | Recommended GPUs |
|---|---|---|---:|
| Blackwell B200 or B300 | EP2 on GPUs 0-1 | SGLang on GPU 0 and training on GPUs 1-2 | 3 |
| Hopper H100 or H200 | EP4 on GPUs 0-3 | SGLang on GPU 0 and training on GPUs 1-4 | 5 |

All GPUs on the host must use the same architecture.

In [ ]:
%%bash
set -euo pipefail

mapfile -t CAPS < <(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | tr -d ' ')
mapfile -t MEMS < <(nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits | tr -d ' ')
GPU_COUNT=${#CAPS[@]}
FIRST_CAP=${CAPS[0]}
FIRST_MEM=${MEMS[0]}

for cap in "${CAPS[@]}"; do
  test "${cap%%.*}" = "${FIRST_CAP%%.*}" || { echo "Mixed GPU architectures are not supported."; exit 1; }
done

MAJOR=${FIRST_CAP%%.*}
if (( MAJOR >= 10 )); then
  HALO_IMAGE=public.ecr.aws/whitecircle/halo:blackwell
  REQUIRED_GPUS=3
  SFT_DEVICES=0,1
  SFT_NPROC=2
  SFT_EP=2
  RL_DEVICES=1,2
  RL_NPROC=2
elif (( MAJOR == 9 )); then
  HALO_IMAGE=public.ecr.aws/whitecircle/halo:hopper
  REQUIRED_GPUS=5
  SFT_DEVICES=0,1,2,3
  SFT_NPROC=4
  SFT_EP=4
  RL_DEVICES=1,2,3,4
  RL_NPROC=4
else
  echo "This guide requires a Hopper or Blackwell GPU. Found compute capability ${FIRST_CAP}."
  exit 1
fi

(( GPU_COUNT >= REQUIRED_GPUS )) || { echo "This layout needs ${REQUIRED_GPUS} GPUs. Found ${GPU_COUNT}."; exit 1; }

cat > /tmp/halo-gemma4-cookbook.env <<EOF
HALO_IMAGE=${HALO_IMAGE}
SGLANG_IMAGE=public.ecr.aws/whitecircle/halo:sglang-0.5.14
SFT_DEVICES=${SFT_DEVICES}
SFT_NPROC=${SFT_NPROC}
SFT_EP=${SFT_EP}
RL_DEVICES=${RL_DEVICES}
RL_NPROC=${RL_NPROC}
EOF

echo "compute capability: ${FIRST_CAP}"
echo "memory per GPU: ${FIRST_MEM} MiB"
echo "training image: ${HALO_IMAGE}"
echo "SFT: ${SFT_NPROC} processes, EP${SFT_EP}, devices ${SFT_DEVICES}"
echo "GRPO: SGLang device 0, ${RL_NPROC} training processes, devices ${RL_DEVICES}"

## Get Halo and the public images

Install the plotting dependency. Then get Halo and create persistent cache and checkpoint directories.

In [ ]:
%pip install --quiet matplotlib

In [ ]:
%%bash
set -euo pipefail
source /tmp/halo-gemma4-cookbook.env

if test -d halo/.git; then
  git -C halo pull --ff-only
  git -C halo submodule update --init --recursive
else
  git clone --recurse-submodules https://github.com/whitecircle/halo
fi

mkdir -p /mnt/hf /mnt/checkpoints /mnt/tmp
docker pull "${HALO_IMAGE}"
docker pull "${SGLANG_IMAGE}"

## Run ten SFT steps

Halo uses DeepEP and grouped GEMM during SFT. It also uses a fused Triton tanh-GeGLU kernel. The run saves a standard Hugging Face checkpoint after step ten.

In [ ]:
%%bash
set -euo pipefail
source /tmp/halo-gemma4-cookbook.env

cat > halo/gemma4-sft-10.yaml <<EOF
model_name_or_path: google/gemma-4-26B-A4B-it
trust_remote_code: true
moe_balancing: none
dataset:
- HuggingFaceH4/ultrachat_200k@train_sft
conversation_field: messages
test_size: 0.01
train_only_on_completions: true
assistant_message_template: "<|turn>model\n"
pad_token: <pad>
expert_parallel_size: ${SFT_EP}
save_sharded_ep: false
use_grouped_gemm: true
attn_implementation: sdpa
use_liger_kernel: true
packing: false
padding_free: false
max_length: 2048
bf16: true
bf16_optimizer: true
per_device_train_batch_size: 1
per_device_eval_batch_size: 1
gradient_accumulation_steps: 1
max_steps: 10
gradient_checkpointing: true
gradient_checkpointing_kwargs:
  use_reentrant: false
optim: adamw_torch_fused
learning_rate: 5.0e-06
lr_scheduler_type: cosine
warmup_steps: 2
max_grad_norm: 1.0
save_strategy: steps
save_steps: 10
eval_strategy: 'no'
save_total_limit: 1
save_only_model: true
output_dir: /mnt/checkpoints/gemma-4-26b-a4b-sft-10
logging_steps: 1
logging_first_step: true
report_to: none
remove_unused_columns: false
dataloader_num_workers: 2
use_peft: false
EOF

In [ ]:
%%bash
set -euo pipefail
source /tmp/halo-gemma4-cookbook.env
SFT_GPU_REQUEST="\"device=${SFT_DEVICES}\""

docker run --rm \
  --name halo-gemma4-sft \
  --gpus "${SFT_GPU_REQUEST}" \
  --ipc=host \
  --shm-size=128g \
  --ulimit memlock=-1 \
  --ulimit stack=67108864 \
  -e HF_HOME=/mnt/hf \
  -e HF_DATASETS_CACHE=/mnt/hf/datasets \
  -e TMPDIR=/mnt/tmp \
  -e HALO_DATA_ROOT=/mnt/tmp \
  -e PYTHONPATH=/workspace \
  -e CUDA_DEVICE_MAX_CONNECTIONS=1 \
  -v "${PWD}/halo":/workspace \
  -v /mnt/hf:/mnt/hf \
  -v /mnt/tmp:/mnt/tmp \
  -v /mnt/checkpoints:/mnt/checkpoints \
  -w /workspace \
  "${HALO_IMAGE}" \
  halo launch sft gemma4-sft-10.yaml -n "${SFT_NPROC}"

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

checkpoint = Path("/mnt/checkpoints/gemma-4-26b-a4b-sft-10/checkpoint-10")
required = ["config.json", "model.safetensors.index.json", "trainer_state.json"]
missing = [name for name in required if not (checkpoint / name).exists()]
if missing:
    raise FileNotFoundError(f"Missing checkpoint files: {missing}")

state = json.loads((checkpoint / "trainer_state.json").read_text())
loss_history = [entry for entry in state["log_history"] if "loss" in entry and "step" in entry]
if not loss_history:
    raise RuntimeError("The checkpoint does not contain SFT loss values.")

steps = [entry["step"] for entry in loss_history]
losses = [entry["loss"] for entry in loss_history]

fig, ax = plt.subplots(figsize=(7, 4), dpi=200)
ax.plot(steps, losses, marker="o", linewidth=2)
ax.set(title="Gemma 4 SFT loss", xlabel="Training step", ylabel="Loss")
ax.set_xticks(steps)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## Run ten environmental GRPO steps with SGLang

SGLang runs on GPU 0. The Halo trainer uses the other GPUs. The two containers use host networking for NCCL weight synchronization.

This GRPO path uses the native Gemma expert layout. Set `expert_parallel_size: 1` and `use_grouped_gemm: false`. SGLang weight synchronization does not support Halo expert distribution for Gemma 4.

The run adds LoRA adapters to the attention projections. It uses AIME 2024 questions and grades the final boxed numeric answer.

In [ ]:
%%writefile halo/gemma4-grpo-sglang-10.yaml
model_name_or_path: /mnt/checkpoints/gemma-4-26b-a4b-sft-10/checkpoint-10
trust_remote_code: true
attn_implementation: sdpa
moe_balancing: none
expert_parallel_size: 1
use_grouped_gemm: false
fsdp_reshard_after_forward: false
rollout_backend: sglang
rollout_server_url: http://localhost:30000
rollout_connection_timeout: 300.0
sync_weights_every_n_steps: 1
train_on_sampled_tokens: true
routing_replay: none
rollout_temperature: 1.0
rollout_top_p: 0.95
rollout_max_tokens: 1024
request_timeout: 300.0
episode_timeout: 600
enable_prefetch: false
num_rollout_workers: 4
max_concurrent_rollouts: 4
env_type: exam_qa
success_reward: 1.0
failure_reward: 0.0
max_turns: 1
environment_kwargs:
  system_prompt: >-
    Solve the problem carefully. You may show your reasoning, but end with exactly one
    final-answer line in the form `Final Answer: \boxed{N}`, where N is the requested
    integer. Do not continue after the boxed answer.
dataset:
- HuggingFaceH4/aime_2024
test_size: null
prompt_field: problem
answer_field: answer
max_prompt_length: 1024
num_generations: 4
beta: 0.0
num_iterations: 1
epsilon: 0.2
scale_rewards: batch
drop_degenerate_groups: true
per_device_train_batch_size: 1
per_device_eval_batch_size: 1
gradient_accumulation_steps: 4
steps_per_generation: 4
max_steps: 10
gradient_checkpointing: true
gradient_checkpointing_kwargs:
  use_reentrant: false
bf16: true
bf16_optimizer: true
fp32_grad_reduce: true
optim: adamw_torch_fused
learning_rate: 1.0e-06
lr_scheduler_type: cosine
warmup_steps: 2
max_grad_norm: 1.0
use_liger_kernel: true
use_peft: true
lora_r: 16
lora_alpha: 32
lora_dropout: 0.05
lora_task_type: CAUSAL_LM
lora_target_modules: '.*language_model\.layers\.[0-9]+\.self_attn\.(q_proj|k_proj|v_proj|o_proj)$'
output_dir: /mnt/checkpoints/gemma-4-26b-a4b-grpo-sglang-10
save_strategy: steps
save_steps: 10
save_total_limit: 1
eval_strategy: 'no'
logging_steps: 1
logging_first_step: true
report_to: none
log_completions: true
save_completions: true
num_completions_to_print: 2
dataloader_num_workers: 0
remove_unused_columns: false
seed: 42

In [ ]:
%%bash
set -euo pipefail
source /tmp/halo-gemma4-cookbook.env
cd halo

SGLANG_IMAGE="${SGLANG_IMAGE}" \
SGLANG_MODEL=/mnt/checkpoints/gemma-4-26b-a4b-sft-10/checkpoint-10 \
SGLANG_MODEL_DIR=/mnt/checkpoints \
SGLANG_CUDA_DEVICES=0 \
SGLANG_PORT=30000 \
SGLANG_GPU_MEM=0.85 \
HF_HOME=/mnt/hf \
docker compose -f docker-compose.sglang.yml up -d

In [ ]:
import time
import urllib.request

health_url = "http://localhost:30000/health"
for attempt in range(60):
    try:
        with urllib.request.urlopen(health_url, timeout=10) as response:
            if response.status == 200:
                print("SGLang is ready.")
                break
    except Exception:
        if attempt == 59:
            raise
        time.sleep(10)

In [ ]:
%%bash
set -euo pipefail
source /tmp/halo-gemma4-cookbook.env
RL_GPU_REQUEST="\"device=${RL_DEVICES}\""

docker run --rm \
  --name halo-gemma4-grpo \
  --gpus "${RL_GPU_REQUEST}" \
  --ipc=host \
  --network=host \
  --shm-size=128g \
  --ulimit memlock=-1 \
  --ulimit stack=67108864 \
  -e HF_HOME=/mnt/hf \
  -e HF_DATASETS_CACHE=/mnt/hf/datasets \
  -e TMPDIR=/mnt/tmp \
  -e RAY_TMPDIR=/dev/shm/ray \
  -e HALO_DATA_ROOT=/mnt/tmp \
  -e PYTHONPATH=/workspace \
  -e DIST_NCCL_TIMEOUT_MINUTES=30 \
  -e NCCL_P2P_DISABLE=1 \
  -e NCCL_SHM_DISABLE=1 \
  -e NCCL_NET=Socket \
  -e NCCL_IB_DISABLE=1 \
  -e NCCL_NET_PLUGIN=none \
  -v "${PWD}/halo":/workspace \
  -v /mnt/hf:/mnt/hf \
  -v /mnt/tmp:/mnt/tmp \
  -v /mnt/checkpoints:/mnt/checkpoints \
  -w /workspace \
  "${HALO_IMAGE}" \
  halo launch environmental-grpo gemma4-grpo-sglang-10.yaml -n "${RL_NPROC}"

In [ ]:
from pathlib import Path

output_dir = Path("/mnt/checkpoints/gemma-4-26b-a4b-grpo-sglang-10")
trainer_state = list(output_dir.glob("checkpoint-10/trainer_state.json"))
completions = list(output_dir.glob("completions/*.parquet"))
if not trainer_state:
    raise FileNotFoundError("The ten-step GRPO checkpoint is missing.")
if not completions:
    raise FileNotFoundError("The GRPO completion records are missing.")
print(trainer_state[0])
print(f"completion files: {len(completions)}")

In [ ]:
%%bash
set -euo pipefail
source /tmp/halo-gemma4-cookbook.env
cd halo
SGLANG_IMAGE="${SGLANG_IMAGE}" \
SGLANG_MODEL=/mnt/checkpoints/gemma-4-26b-a4b-sft-10/checkpoint-10 \
SGLANG_MODEL_DIR=/mnt/checkpoints \
docker compose -f docker-compose.sglang.yml down